## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:刘译幡


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [1]:
## add your code here
import java.io.*;
import java.util.*;

public class Main {
    static class Move {
        int type, val;
        Move(int type, int val) {
            this.type = type;
            this.val = val;
        }
    }

    static class PermutationRestorer {
        int n, a, b, l;
        int[] board;
        int[] pos;
        List<Move> moves;

        public PermutationRestorer(int size, int val_a, int val_b, int[] initial_board) {
            this.n = size;
            this.a = val_a;
            this.b = val_b;
            this.board = initial_board.clone();
            this.pos = new int[n];
            this.moves = new ArrayList<>();

            int diff = (a - b + n) % n;
            this.l = diff & -diff;
            if (this.l == 0) this.l = n;

            for (int i = 0; i < n; i++) this.pos[board[i]] = i;
        }

        void apply_add(int x) {
            if (x == 0) return;
            moves.add(new Move(2, x));
            for (int i = 0; i < n; i++) {
                board[i] = (board[i] + x) % n;
                pos[board[i]] = i;
            }
        }

        void apply_xor(int x) {
            if (x == 0) return;
            moves.add(new Move(1, x));
            for (int i = 0; i < n; i++) {
                board[i] ^= x;
                pos[board[i]] = i;
            }
        }

        void append_add(int x) {
            if (x != 0) moves.add(new Move(2, x));
        }

        void append_xor(int x) {
            if (x != 0) moves.add(new Move(1, x));
        }

        int[] compute_target_positions(int u, int v) {
            int delta = (v - u + 2 * n - l) % n;
            int pu = 0, pv = 0;
            int step = n / 2;
            while (step >= 2 * l) {
                if (delta >= step) {
                    delta -= step;
                    pv += step / 2;
                } else {
                    pu += step / 2;
                }
                step /= 2;
            }
            pu += n / 2;
            pu += u & (l - 1);
            pv += u & (l - 1);
            return new int[]{pu, pv};
        }

        void swap_two_values(int c, int d) {
            if (c == d) return;
            int block_c = (c / l) % 2;
            int block_d = (d / l) % 2;
            if (block_c == block_d) {
                int pivot = (block_c == 0) ? ((c & (l - 1)) + l) : (c & (l - 1));
                swap_two_values(c, pivot);
                swap_two_values(d, pivot);
                swap_two_values(c, pivot);
                return;
            }

            int[] p_ab = compute_target_positions(a, b);
            int[] p_cd = compute_target_positions(c, d);
            int pa = p_ab[0], pb = p_ab[1];
            int pc = p_cd[0], pd = p_cd[1];

            append_add((pc - c + n) % n);
            append_xor(pc ^ pa);
            append_add((a - pa + n) % n);
            moves.add(new Move(0, 0));
            append_add((pa - a + n) % n);
            append_xor(pc ^ pa);
            append_add((c - pc + n) % n);

            int pc_idx = pos[c];
            int pd_idx = pos[d];
            int temp = board[pc_idx];
            board[pc_idx] = board[pd_idx];
            board[pd_idx] = temp;
            pos[c] = pd_idx;
            pos[d] = pc_idx;
        }

        boolean resolve_low_order(int[] sequence, int m, List<Integer> out_ops) {
            boolean[] seen = new boolean[m];
            for (int x : sequence) {
                if (x < 0 || x >= m || seen[x]) return false;
                seen[x] = true;
            }
            if (m == 1) return true;

            int half = m / 2;
            int[] even_part = new int[half];
            int[] odd_part = new int[half];
            for (int i = 0; i < half; i++) {
                even_part[i] = sequence[2 * i] / 2;
                odd_part[i] = sequence[2 * i + 1] / 2;
            }

            List<Integer> even_ops = new ArrayList<>();
            List<Integer> odd_ops = new ArrayList<>();
            if (!resolve_low_order(even_part, half, even_ops)) return false;
            if (!resolve_low_order(odd_part, half, odd_ops)) return false;

            List<Integer> combined_ops = new ArrayList<>();
            if (sequence[0] % 2 != 0) {
                combined_ops.add(m == 2 ? 1 : -1);
            }

            int tb = 0;
            for (int op : even_ops) {
                if (op > 0) {
                    combined_ops.add(-1);
                    combined_ops.add(1);
                } else {
                    combined_ops.add(op * 2);
                    tb ^= (-op * 2);
                }
            }
            if (tb != 0) combined_ops.add(-tb);

            int tc = 0;
            for (int op : odd_ops) {
                if (op > 0) {
                    combined_ops.add(1);
                    combined_ops.add(-1);
                } else {
                    combined_ops.add(op * 2);
                    tc ^= (-op * 2);
                }
            }

            if ((tc & half) != (tb & half)) {
                for (int i = 0; i < half / 2; i++) {
                    combined_ops.add(-1);
                    combined_ops.add(1);
                }
            }
            if (tb >= half) tb -= half;
            if (tc >= half) tc -= half;
            if (tb != tc) return false;

            for (int op : combined_ops) {
                if (out_ops.isEmpty()) {
                    out_ops.add(op);
                } else if (op < 0 && out_ops.get(out_ops.size() - 1) < 0) {
                    int last = out_ops.remove(out_ops.size() - 1);
                    int new_xor = -((-last) ^ (-op));
                    if (new_xor != 0) out_ops.add(new_xor);
                } else {
                    out_ops.add(op);
                }
            }
            return true;
        }

        boolean restore() {
            if (l > 1) {
                int[] low_bits = new int[l];
                for (int i = 0; i < l; i++) low_bits[i] = board[i] & (l - 1);
                List<Integer> ops = new ArrayList<>();
                if (!resolve_low_order(low_bits, l, ops)) return false;
                for (int op : ops) {
                    if (op > 0) apply_add(op);
                    else apply_xor(-op);
                }
            }

            for (int rem = 0; rem < l; rem++) {
                List<Integer> actual = new ArrayList<>();
                for (int idx = rem; idx < n; idx += l) actual.add(board[idx]);
                Collections.sort(actual);
                int expected = rem;
                for (int x : actual) {
                    if (x != expected) return false;
                    expected += l;
                }
            }

            for (int rem = 0; rem < l; rem++) {
                for (int idx = rem; idx < n; idx += l) {
                    while (board[idx] != idx) {
                        swap_two_values(idx, board[idx]);
                    }
                }
            }

            for (int i = 0; i < n; i++) {
                if (board[i] != i) return false;
            }
            return true;
        }

        void printSolution(PrintWriter out) {
            out.println(moves.size());
            for (Move op : moves) {
                if (op.type == 0) out.println("0");
                else out.println(op.type + " " + op.val);
            }
        }
    }

    static class FastScanner {
        BufferedReader br;
        StringTokenizer st;
        public FastScanner() { br = new BufferedReader(new InputStreamReader(System.in)); }
        String next() {
            while (st == null || !st.hasMoreElements()) {
                try { st = new StringTokenizer(br.readLine()); }
                catch (IOException e) { e.printStackTrace(); }
            }
            return st.nextToken();
        }
        int nextInt() { return Integer.parseInt(next()); }
    }

    public static void main(String[] args) {
        FastScanner sc = new FastScanner();
        PrintWriter out = new PrintWriter(new BufferedOutputStream(System.out));
        try {
            int n = sc.nextInt();
            int a = sc.nextInt();
            int b = sc.nextInt();
            int[] init = new int[n];
            for (int i = 0; i < n; i++) init[i] = sc.nextInt();

            PermutationRestorer solver = new PermutationRestorer(n, a, b, init);
            if (!solver.restore()) {
                out.println("-1");
            } else {
                solver.printSolution(out);
            }
        } catch (Exception e) {
        } finally {
            out.flush();
            out.close();
        }
    }
}

## B 长跑

In [ ]:
## add your code here
import java.util.*;

public class Main {
    static class Node {
        int a, b;
        Node(int a, int b) {
            this.a = a;
            this.b = b;
        }
    }

    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        
        while (sc.hasNext()) {
            int N = sc.nextInt();
            int L = sc.nextInt();
            int max_energy = sc.nextInt(); 
            int S = sc.nextInt();

            List<Node> nodes = new ArrayList<>();
            nodes.add(new Node(0, 0)); 
            
            for (int i = 0; i < N; i++) {
                int a = sc.nextInt();
                int b = sc.nextInt();
                nodes.add(new Node(a, b));
            }

            Collections.sort(nodes, (node1, node2) -> node1.a - node2.a);

            int[] dp = new int[nodes.size()];
            Arrays.fill(dp, -1);
            
            dp[0] = S;

            boolean possible = false;
            
            if (max_energy >= L) {
                possible = true;
            } else {
                for (int i = 1; i < nodes.size(); i++) {
                    for (int j = 0; j < i; j++) {
                        if (dp[j] >= 0 && nodes.get(i).a - nodes.get(j).a <= max_energy) {
                            int remain = dp[j] - nodes.get(i).b;
                            if (remain >= 0) {
                                dp[i] = Math.max(dp[i], remain);
                            }
                        }
                    }
                    
                    if (dp[i] >= 0 && L - nodes.get(i).a <= max_energy) {
                        possible = true;
                        break;
                    }
                }
            }

            System.out.println(possible ? "Yes" : "No");
        }
        sc.close();
    }
}

## C 最长回文

In [ ]:
## add your code here
import java.io.BufferedReader;
import java.io.IOException;
import java.io.InputStreamReader;

public class Main {
    static final long MOD1 = 1000000007L;
    static final long MOD2 = 1000000009L;
    static final long BASE1 = 313L;
    static final long BASE2 = 317L;

    static class StringHash {
        long[] h1, h2, p1, p2;

        public StringHash(String s) {
            int n = s.length();
            h1 = new long[n + 1];
            h2 = new long[n + 1];
            p1 = new long[n + 1];
            p2 = new long[n + 1];
            p1[0] = 1;
            p2[0] = 1;
            for (int i = 0; i < n; ++i) {
                h1[i + 1] = (h1[i] * BASE1 + s.charAt(i)) % MOD1;
                h2[i + 1] = (h2[i] * BASE2 + s.charAt(i)) % MOD2;
                p1[i + 1] = (p1[i] * BASE1) % MOD1;
                p2[i + 1] = (p2[i] * BASE2) % MOD2;
            }
        }
    }

    static long getHash(StringHash H, int l, int r) {
        if (l > r) return 0;
        long v1 = (H.h1[r] - H.h1[l - 1] * H.p1[r - l + 1]) % MOD1;
        if (v1 < 0) v1 += MOD1;
        long v2 = (H.h2[r] - H.h2[l - 1] * H.p2[r - l + 1]) % MOD2;
        if (v2 < 0) v2 += MOD2;
        return (v1 << 32) | v2;
    }

    static int get_lcp(StringHash H1, int i, StringHash H2, int j, int n) {
        int max_len = Math.min(n - i + 1, n - j + 1);
        if (max_len <= 0) return 0;
        int low = 1, high = max_len, ans = 0;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (getHash(H1, i, i + mid - 1) == getHash(H2, j, j + mid - 1)) {
                ans = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        return ans;
    }

    static int[] manacher(String s) {
        StringBuilder t = new StringBuilder("^#");
        for (int i = 0; i < s.length(); ++i) {
            t.append(s.charAt(i)).append("#");
        }
        t.append("$");
        int m = t.length();
        int[] p = new int[m];
        int c = 0, r = 0;
        for (int i = 1; i < m - 1; ++i) {
            int i_mirror = 2 * c - i;
            if (r > i) p[i] = Math.min(r - i, p[i_mirror]);
            while (t.charAt(i + 1 + p[i]) == t.charAt(i - 1 - p[i])) p[i]++;
            if (i + p[i] > r) {
                c = i;
                r = i + p[i];
            }
        }
        return p;
    }

    public static void main(String[] args) throws IOException {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        String line = br.readLine();
        if (line == null || line.trim().isEmpty()) return;
        
        int n = Integer.parseInt(line.trim());
        String A = br.readLine().trim();
        String B = br.readLine().trim();

        String A_rev = new StringBuilder(A).reverse().toString();

        StringHash HA_rev = new StringHash(A_rev);
        StringHash HB = new StringHash(B);

        int[] pA = manacher(A);
        int[] pB = manacher(B);

        int max_pal_len = 0;

        for (int i = 1; i <= 2 * n + 1; ++i) {
            int L, R;
            if (i % 2 == 0) {
                int j = i / 2;
                int r_s = (pA[i] - 1) / 2;
                L = j - r_s;
                R = j + r_s;
            } else {
                int j = (i - 1) / 2;
                int r_s = pA[i] / 2;
                L = j - r_s + 1;
                R = j + r_s;
            }
            int k = get_lcp(HA_rev, n - L + 2, HB, R, n);
            max_pal_len = Math.max(max_pal_len, R - L + 1 + 2 * k);
        }

        for (int i = 1; i <= 2 * n + 1; ++i) {
            int L, R;
            if (i % 2 == 0) {
                int j = i / 2;
                int r_s = (pB[i] - 1) / 2;
                L = j - r_s;
                R = j + r_s;
            } else {
                int j = (i - 1) / 2;
                int r_s = pB[i] / 2;
                L = j - r_s + 1;
                R = j + r_s;
            }
            int k = get_lcp(HA_rev, n - L + 1, HB, R + 1, n);
            max_pal_len = Math.max(max_pal_len, R - L + 1 + 2 * k);
        }

        System.out.println(max_pal_len);
    }
}

## D 优惠券

In [ ]:
## add your code here
import java.util.*;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        while (sc.hasNextInt()) {
            int m = sc.nextInt();
            
            if (m == 0) {
                System.out.println("-1");
                continue;
            }
            
            Set<Integer> heldCoupons = new HashSet<>();
            Map<Integer, Integer> lastIdx = new HashMap<>();
            TreeSet<Integer> questions = new TreeSet<>();
            
            int errorLine = -1;
            
            for (int i = 1; i <= m; i++) {
                String op = sc.next();
                
                boolean isIO = op.equals("I") || op.equals("O");
                int x = 0;
                if (isIO) {
                    x = sc.nextInt();
                }
                
                if (errorLine != -1) continue;
                
                if (op.equals("?") || op.equals("？")) {
                    questions.add(i);
                } else if (op.equals("I")) {
                    if (heldCoupons.contains(x)) {
                        int prev = lastIdx.getOrDefault(x, 0);
                        Integer qIdx = questions.higher(prev);
                        if (qIdx == null) {
                            errorLine = i;
                        } else {
                            questions.remove(qIdx);
                        }
                    }
                    heldCoupons.add(x);
                    lastIdx.put(x, i);
                } else if (op.equals("O")) {
                    if (!heldCoupons.contains(x)) {
                        int prev = lastIdx.getOrDefault(x, 0);
                        Integer qIdx = questions.higher(prev);
                        if (qIdx == null) {
                            errorLine = i;
                        } else {
                            questions.remove(qIdx);
                        }
                    } else {
                        heldCoupons.remove(x);
                    }
                    lastIdx.put(x, i);
                }
            }
            System.out.println(errorLine);
        }
    }
}

## E 任意点

In [ ]:
## add your code here
import java.util.Scanner;

public class Main {
    static int[] parent;

    public static int find(int i) {
        if (parent[i] == i)
            return i;
        return parent[i] = find(parent[i]);
    }

    public static void union(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI != rootJ) {
            parent[rootI] = rootJ;
        }
    }

    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        if (!sc.hasNextInt()) return;
        
        int n = sc.nextInt();
        int[] x = new int[n];
        int[] y = new int[n];
        parent = new int[n];

        for (int i = 0; i < n; i++) {
            x[i] = sc.nextInt();
            y[i] = sc.nextInt();
            parent[i] = i;
        }

        for (int i = 0; i < n; i++) {
            for (int j = i + 1; j < n; j++) {
                if (x[i] == x[j] || y[i] == y[j]) {
                    union(i, j);
                }
            }
        }

        int count = 0;
        for (int i = 0; i < n; i++) {
            if (parent[i] == i) {
                count++;
            }
        }

        System.out.println(count - 1);
        sc.close();
    }
}

## F 通配符匹配

In [ ]:
## add your code here
import java.io.*;
import java.util.*;

public class Main {
    static long MOD1 = 1000000007L;
    static long MOD2 = 1000000009L;
    static long B1 = 313L;
    static long B2 = 317L;
    static int MAXL = 100005;
    static long[] pb1 = new long[MAXL];
    static long[] pb2 = new long[MAXL];

    static {
        pb1[0] = 1; pb2[0] = 1;
        for (int i = 1; i < MAXL; i++) {
            pb1[i] = (pb1[i-1] * B1) % MOD1;
            pb2[i] = (pb2[i-1] * B2) % MOD2;
        }
    }

    static class Segment {
        String s;
        long hp1, hp2;
        List<Integer> q_pos = new ArrayList<>();

        Segment(String s) {
            this.s = s;
            int L = s.length();
            for (int i = 0; i < L; i++) {
                if (s.charAt(i) == '?') {
                    q_pos.add(i);
                    hp1 = (hp1 * B1) % MOD1;
                    hp2 = (hp2 * B2) % MOD2;
                } else {
                    long val = s.charAt(i);
                    hp1 = (hp1 * B1 + val) % MOD1;
                    hp2 = (hp2 * B2 + val) % MOD2;
                }
            }
        }
    }

    static boolean match_seg(String S, String P_seg, int offset) {
        for (int i = 0; i < P_seg.length(); i++) {
            if (P_seg.charAt(i) != '?' && P_seg.charAt(i) != S.charAt(offset + i)) return false;
        }
        return true;
    }

    static int find_first_match(String S, int s_start, int s_end, Segment seg) {
        int L = seg.s.length();
        if (L == 0) return s_start;
        if (s_end - s_start < L) return -1;

        long hs1 = 0, hs2 = 0;
        for (int i = 0; i < L; i++) {
            long val = S.charAt(s_start + i);
            hs1 = (hs1 * B1 + val) % MOD1;
            hs2 = (hs2 * B2 + val) % MOD2;
        }

        for (int pos = s_start; pos <= s_end - L; pos++) {
            long cur_hs1 = hs1, cur_hs2 = hs2;
            for (int q : seg.q_pos) {
                long val = S.charAt(pos + q);
                cur_hs1 = (cur_hs1 - val * pb1[L - 1 - q] % MOD1 + MOD1) % MOD1;
                cur_hs2 = (cur_hs2 - val * pb2[L - 1 - q] % MOD2 + MOD2) % MOD2;
            }

            if (cur_hs1 == seg.hp1 && cur_hs2 == seg.hp2) {
                if (match_seg(S, seg.s, pos)) return pos;
            }

            if (pos < s_end - L) {
                long val_out = S.charAt(pos);
                long val_in = S.charAt(pos + L);
                hs1 = (hs1 - val_out * pb1[L - 1] % MOD1 + MOD1) % MOD1;
                hs1 = (hs1 * B1 + val_in) % MOD1;

                hs2 = (hs2 - val_out * pb2[L - 1] % MOD2 + MOD2) % MOD2;
                hs2 = (hs2 * B2 + val_in) % MOD2;
            }
        }
        return -1;
    }

    static class FastScanner {
        BufferedReader br;
        StringTokenizer st;
        public FastScanner() { br = new BufferedReader(new InputStreamReader(System.in)); }
        String next() {
            while (st == null || !st.hasMoreElements()) {
                try {
                    String line = br.readLine();
                    if (line == null) return null;
                    st = new StringTokenizer(line);
                } catch (IOException e) { e.printStackTrace(); }
            }
            return st.nextToken();
        }
        int nextInt() { return Integer.parseInt(next()); }
    }

    public static void main(String[] args) {
        FastScanner sc = new FastScanner();
        BufferedWriter bw = new BufferedWriter(new OutputStreamWriter(System.out));
        try {
            String pattern;
            while ((pattern = sc.next()) != null) {
                int n = sc.nextInt();

                List<Segment> segs = new ArrayList<>();
                StringBuilder current_seg = new StringBuilder();
                for (char c : pattern.toCharArray()) {
                    if (c == '*') {
                        segs.add(new Segment(current_seg.toString()));
                        current_seg.setLength(0);
                    } else {
                        current_seg.append(c);
                    }
                }
                segs.add(new Segment(current_seg.toString()));
                int m = segs.size() - 1;

                for (int k = 0; k < n; k++) {
                    String S = sc.next();

                    if (m == 0) {
                        if (S.length() == segs.get(0).s.length() && match_seg(S, segs.get(0).s, 0)) {
                            bw.write("YES\n");
                        } else {
                            bw.write("NO\n");
                        }
                        continue;
                    }

                    if (S.length() < segs.get(0).s.length() + segs.get(m).s.length()) {
                        bw.write("NO\n");
                        continue;
                    }

                    if (!match_seg(S, segs.get(0).s, 0)) {
                        bw.write("NO\n");
                        continue;
                    }

                    int s_idx = segs.get(0).s.length();
                    int s_end = S.length() - segs.get(m).s.length();

                    if (!match_seg(S, segs.get(m).s, s_end)) {
                        bw.write("NO\n");
                        continue;
                    }

                    boolean possible = true;
                    for (int i = 1; i < m; i++) {
                        int pos = find_first_match(S, s_idx, s_end, segs.get(i));
                        if (pos == -1) {
                            possible = false;
                            break;
                        }
                        s_idx = pos + segs.get(i).s.length();
                    }

                    if (possible) bw.write("YES\n");
                    else bw.write("NO\n");
                }
            }
            bw.flush();
        } catch (Exception e) {
            e.printStackTrace();
        }
    }
}

## G 汉诺塔

In [ ]:
## add your code here
import java.util.Scanner;
import java.util.HashMap;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        
        if (!sc.hasNextInt()) {
            return;
        }
        int n = sc.nextInt();

        HashMap<String, Integer> score = new HashMap<>();
        for (int i = 0; i < 6; i++) {
            String s = sc.next();
            score.put(s, 5 - i);
        }

        int[][] D = new int[35][3];
        long[][] S = new long[35][3];

        for (int X = 0; X < 3; X++) {
            char from = (char) ('A' + X);
            char to1 = (char) ('A' + (X + 1) % 3);
            char to2 = (char) ('A' + (X + 2) % 3);

            String move1 = "" + from + to1;
            String move2 = "" + from + to2;

            if (score.get(move1) > score.get(move2)) {
                D[1][X] = (X + 1) % 3;
            } else {
                D[1][X] = (X + 2) % 3;
            }
            S[1][X] = 1;
        }

        for (int i = 2; i <= n; i++) {
            for (int X = 0; X < 3; X++) {
                int Z = D[i - 1][X];
                int Y = 3 - X - Z; 

                int W = D[i - 1][Z];

                if (W == Y) {
                    D[i][X] = Y;
                    S[i][X] = S[i - 1][X] + 1 + S[i - 1][Z];
                } else if (W == X) {
                    D[i][X] = Z;
                    S[i][X] = S[i - 1][X] + 1 + S[i - 1][Z] + 1 + S[i - 1][X];
                }
            }
        }

        System.out.println(S[n][0]);
        
        sc.close();
    }
}

## H 马步距离

In [ ]:
## add your code here
import java.util.Scanner;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        
        if (sc.hasNextLong()) {
            long x1 = sc.nextLong();
            long y1 = sc.nextLong();
            long x2 = sc.nextLong();
            long y2 = sc.nextLong();
            
            System.out.println(minMoves(x1, y1, x2, y2));
        }
        sc.close();
    }

    public static long minMoves(long x1, long y1, long x2, long y2) {
        long x = Math.abs(x1 - x2);
        long y = Math.abs(y1 - y2);
        
        if (x < y) {
            long temp = x;
            x = y;
            y = temp;
        }
        
        if (x == 1 && y == 0) return 3; 
        if (x == 2 && y == 2) return 4; 
        
        long ans = Math.max((x + 1) / 2, (x + y + 2) / 3);
        
        if (ans % 2 != (x + y) % 2) {
            ans++;
        }
        
        return ans;
    }
}

## I 直方图最大矩形

In [ ]:
## add your code here
import java.util.*;

public class Solution {
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * @param heights int整型一维数组 
     * @return int整型
     */
    public int largestRectangleArea (int[] heights) {
        if (heights == null || heights.length == 0) {
            return 0;
        }

        int[] newHeights = new int[heights.length + 2];
        newHeights[0] = 0;
        newHeights[newHeights.length - 1] = 0;
        System.arraycopy(heights, 0, newHeights, 1, heights.length);

        Deque<Integer> stack = new ArrayDeque<>();
        int maxArea = 0;

        for (int i = 0; i < newHeights.length; i++) {
            while (!stack.isEmpty() && newHeights[i] < newHeights[stack.peek()]) {
                int curHeightIndex = stack.pop();
                int h = newHeights[curHeightIndex];
                
                int leftBound = stack.peek();
                int w = i - leftBound - 1;
                
                maxArea = Math.max(maxArea, h * w);
            }
            stack.push(i);
        }

        return maxArea;
    }
}

## J 消防局的设立

In [ ]:
## add your code here
import java.util.*;

public class Main {
    static List<Integer>[] adj;
    static int[] parent;
    static int[] depth;
    static boolean[] covered;
    static boolean[] hasStation;
    static Integer[] nodes;

    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        if (!sc.hasNextInt()) return;
        int n = sc.nextInt();
        
        adj = new ArrayList[n + 1];
        for (int i = 1; i <= n; i++) adj[i] = new ArrayList<>();
        parent = new int[n + 1];
        depth = new int[n + 1];
        covered = new boolean[n + 1];
        hasStation = new boolean[n + 1];
        
        for (int i = 2; i <= n; i++) {
            int u = i;
            int v = sc.nextInt();
            adj[u].add(v);
            adj[v].add(u);
        }

        bfs(1, n);

        nodes = new Integer[n];
        for (int i = 0; i < n; i++) nodes[i] = i + 1;
        Arrays.sort(nodes, (a, b) -> depth[b] - depth[a]);

        int count = 0;

        for (int u : nodes) {
            if (!isCovered(u)) {
                int target = parent[u] != 0 ? parent[parent[u]] : 1;
                if (target == 0) target = 1; 
                
                if (!hasStation[target]) {
                    placeStation(target);
                    count++;
                }
            }
        }
        System.out.println(count);
    }

    static void bfs(int start, int n) {
        Queue<Integer> q = new LinkedList<>();
        q.add(start);
        depth[start] = 1;
        while (!q.isEmpty()) {
            int u = q.poll();
            for (int v : adj[u]) {
                if (depth[v] == 0) {
                    depth[v] = depth[u] + 1;
                    parent[v] = u;
                    q.add(v);
                }
            }
        }
    }

    static boolean isCovered(int u) {
        if (covered[u]) return true;
        return covered[u];
    }

    static void placeStation(int u) {
        hasStation[u] = true;
        markCovered(u, 0, -1);
    }

    static void markCovered(int u, int d, int p) {
        covered[u] = true;
        if (d < 2) {
            for (int v : adj[u]) {
                if (v != p) {
                    markCovered(v, d + 1, u);
                }
            }
        }
    }
}